In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json

In [2]:
df = pd.read_csv("/kaggle/input/datasets/amitabhajoy/bengaluru-house-price-data/Bengaluru_House_Data.csv")

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
df.shape

df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [4]:
df = df.drop('society', axis=1)

df.shape

(13320, 8)

In [5]:
df = df.dropna()

df.shape

(12710, 8)

In [6]:
df['bhk'] = df['size'].apply(
    lambda x: int(x.split(' ')[0])
)

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07,2
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00,4
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00,3
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00,3
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00,2


In [7]:
def convert_sqft_to_num(x):

    tokens = x.split('-')

    if len(tokens) == 2:
        return (
            float(tokens[0]) +
            float(tokens[1])
        ) / 2

    try:
        return float(x)

    except:
        return None

In [8]:
df['total_sqft'] = df['total_sqft'].apply(
    convert_sqft_to_num
)

In [9]:
df = df.dropna()

In [10]:
df['price_per_sqft'] = (
    df['price'] * 100000
) / df['total_sqft']

In [12]:
df = df[
    df['price_per_sqft'] <= 100000
]

In [13]:
df = df[
    df['total_sqft'] / df['bhk'] >= 300
]

In [14]:
df['location'] = df['location'].apply(
    lambda x: x.strip()
)

In [15]:
location_stats = df['location'].value_counts()

In [16]:
location_stats_less_than_10 = location_stats[
    location_stats <= 10
]

In [17]:
df['location'] = df['location'].apply(
    lambda x:
    'other'
    if x in location_stats_less_than_10
    else x
)

In [18]:
df = df.drop(
    [
        'size',
        'area_type',
        'availability',
        'price_per_sqft'
    ],
    axis=1
)

In [19]:
dummies = pd.get_dummies(df['location'])

In [20]:
df = pd.concat(
    [
        df,
        dummies.drop('other', axis=1)
    ],
    axis=1
)

In [21]:
df = df.drop('location', axis=1)

In [24]:
X = df.drop('price', axis=1)

y = df['price']

In [25]:
X.select_dtypes(include=['object']).columns

Index([], dtype='object')

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [27]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(
    X_train,
    y_train
)

LinearRegression()

In [28]:
model.score(
    X_test,
    y_test
)

0.4975407615679108

In [30]:
with open(
    'house_price_model.pkl',
    'wb'
) as f:

    pickle.dump(
        model,
        f
    )

In [31]:
columns = {
    "data_columns":
    X.columns.tolist()
}

In [32]:
with open(
    "columns.json",
    "w"
) as f:

    json.dump(
        columns,
        f
    )